# Phase 1 Colab runner: where does forgetting live?

All logic lives in `src/`. Runtime: GPU (T4).

Two separate experiments, as in the plan: **A. Flowers102** as the target, **B. EuroSAT** as the target. Each fine-tunes its own models (3 seeds), then reverts / retains each component.

**Safe to stop at any time, and safe to use Run all:**
* Every result JSON is written straight to Drive (`MyDrive/clip-forgetting/results/phase1/`), so it survives a runtime reset even without a push.
* A fine-tune is skipped when a valid checkpoint for that seed is already in Drive. A broken checkpoint is renamed to `.invalid`, never overwritten.
* Component runs continue from the conditions already saved in Drive.
* Every step pushes its results to GitHub right after it finishes. If a step fails, the notebook stops there.

In a new session: run the two **Setup** cells, then continue (or Run all).

## Setup (every session)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os

REPO = 'sohilkhan-2611/-clip-zero-shot-forgetting'
BRANCH = 'phase-1'
CKPT_DIR = '/content/drive/MyDrive/clip-forgetting/checkpoints'
RESULTS_DRIVE = '/content/drive/MyDrive/clip-forgetting/results/phase1'   # survives runtime resets
DATA_DIR = '/content/data'   # local disk, re-downloaded each session
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DRIVE, exist_ok=True)
token = userdata.get('GITHB-TOKEN')

In [ ]:
if not os.path.exists('/content/repo'):
    !git clone -q -b {BRANCH} https://{token}@github.com/{REPO}.git /content/repo
%cd /content/repo
!git pull -q
!pip install -q -e .[dev]
!nvidia-smi --query-gpu=name,memory.total --format=csv

from src import runner as R
KEYS = R.expected_trainable_keys(6)   # the 72 tensors a last-6-blocks fine-tune saves

def ckpt_path(target, seed):
    return f'{CKPT_DIR}/finetune_{target}_vitb16_seed{seed}.pt'

def finetune(target, seed):
    """Train one seed, unless a valid checkpoint already exists. Result JSON goes to Drive."""
    if R.should_train(ckpt_path(target, seed), KEYS):
        R.run(f'python -m src.finetune --config configs/finetune_{target}_vitb16.yaml --set seed={seed} '
              f'data.root={DATA_DIR} output.checkpoint_dir={CKPT_DIR} output.results_dir={RESULTS_DRIVE}')

def components(target, seed):
    """Revert / retain for one checkpoint. Continues from conditions already saved in Drive."""
    ok, reason = R.checkpoint_status(ckpt_path(target, seed), KEYS)
    if not ok:
        raise RuntimeError(f'checkpoint for {target} seed {seed} is {reason}; run its fine-tune first')
    R.run(f'python -m src.components --config configs/phase1_components_{target}.yaml --set '
          f'data.root={DATA_DIR} checkpoints=[{ckpt_path(target, seed)}] output.results_dir={RESULTS_DRIVE}')

def aggregate(target):
    R.run(f"python -m src.aggregate_phase1 --inputs '{RESULTS_DRIVE}/components_{target}_*.json' "
          f"--out {RESULTS_DRIVE}/summary_components_{target}.json")

def push(message):
    """Copy Drive results into the repo, then commit and push to GitHub."""
    print('copied from Drive:', R.sync_json(RESULTS_DRIVE, 'results/phase1'))
    R.run("git config user.name 'colab-runner' && git config user.email 'colab-runner@users.noreply.github.com'")
    R.run('git add results/phase1/')
    R.run(f'git diff --cached --quiet || git commit -q -m "{message}"')
    R.run(f'git pull -q --rebase origin {BRANCH}')
    R.run(f'git push -q origin {BRANCH}')
    R.run('git status -sb | head -1 && git log --oneline -1')

for t in ('flowers', 'eurosat'):
    for s in (0, 1, 2):
        print(t, s, R.checkpoint_status(ckpt_path(t, s), KEYS)[1])
print('results in Drive:', sorted(os.listdir(RESULTS_DRIVE)))

## A1. Flowers102: fine-tune seeds 1 and 2
Seed 0 comes from Phase 0. A seed whose checkpoint is already valid is skipped (seed 1 from last night will be). Each run takes roughly 35 minutes (Phase 0 timing).

In [ ]:
finetune('flowers', 1)
push('Phase 1: Flowers102 fine-tune, seed 1')

In [ ]:
finetune('flowers', 2)
push('Phase 1: Flowers102 fine-tune, seed 2')

## A2. Flowers102: revert / retain per component
One checkpoint per cell: 10 conditions (pre, ft, 4 revert, 4 retain) on Flowers (2,000 images) and the five held-out sets. Saved to Drive after every condition; rerun a cell and it continues where it stopped.

Checks: seed 0 `ft` should equal the Phase 0 fine-tune exactly. Seed 1 `ft` restores the numbers whose JSON was lost (Flowers 0.9695).

In [ ]:
components('flowers', 0)
push('Phase 1: Flowers102 components, seed 0')

In [ ]:
components('flowers', 1)
push('Phase 1: Flowers102 components, seed 1')

In [ ]:
components('flowers', 2)
push('Phase 1: Flowers102 components, seed 2')

## A3. Flowers102: table over seeds

In [ ]:
aggregate('flowers')
push('Phase 1: Flowers102 summary table over seeds')

## B1. EuroSAT: fine-tune seeds 0, 1 and 2
EuroSAT has about 10 times more training images than Flowers (21,600 vs 2,040), so each run takes longer. Here EuroSAT is the target and Flowers becomes a held-out set.

In [ ]:
finetune('eurosat', 0)
push('Phase 1: EuroSAT fine-tune, seed 0')

In [ ]:
finetune('eurosat', 1)
push('Phase 1: EuroSAT fine-tune, seed 1')

In [ ]:
finetune('eurosat', 2)
push('Phase 1: EuroSAT fine-tune, seed 2')

## B2. EuroSAT: revert / retain per component

In [ ]:
components('eurosat', 0)
push('Phase 1: EuroSAT components, seed 0')

In [ ]:
components('eurosat', 1)
push('Phase 1: EuroSAT components, seed 1')

In [ ]:
components('eurosat', 2)
push('Phase 1: EuroSAT components, seed 2')

## B3. EuroSAT: table over seeds

In [ ]:
aggregate('eurosat')
push('Phase 1: EuroSAT summary table over seeds')